In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'DET': ['Kevin Huerter', 'Caris LeVert']}

Out Players:
{'OKC': ['Jalen Williams'], 'LAL': ['Luka Doncic']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 4 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,TOP_PLAYER_ACTIVE,SECOND_PLAYER_ACTIVE,THIRD_PLAYER_ACTIVE,name
27987,NaN,NaN,NaN,2025-26,1630194,Paul Reed,Paul,1610612765,DET,Detroit Pistons,42500203,2026-05-09,DET @ CLE,L,9.766667,4,4,1.000,0,0,0.000,3,3,1.000,2,1,3,0,1,0,0,0,2,2,11,11,13.6,0,0,14.0,1,9:46,1,150.9,160.0,160.0,106.3,100.0,100.0,44.7,60.0,60.0,0.000,0.00,0.0,0.400,0.125,0.231,16.7,15.8,1.000,1.034,0.261,0.261,100.65,100.75,83.96,100.75,0.208,20,4.0,4.0,NaN,4.18,0.77,2.0,1.0,3.0,16.0,0.0,0.0,9.0,0.0,0.0,0.000,4.0,4.0,1.000,1.0,1.0,1.00,41,91,0.451,9,25,0.360,18,22,0.818,17,23,40,23,16.0,12,4,7,25,17,109,-7.0,109.3,114.7,120.4,119.6,-11.1,-4.9,0.561,1.44,16.5,0.404,0.722,0.534,0.168,0.500,0.541,98.0,96.0,80.00,95,0.447,1610612739,CLE,Cleveland Cavaliers,43,74,0.581,12,32,0.375,18,28,0.643,5,28,33,22,15.0,3,7,4,17,25,116,7.0,120.4,119.6,109.3,114.7,11.1,4.9,0.512,1.47,17.3,0.278,0.596,0.466,0.155,0.662,0.672,98.0,96.0,80.00,97,0.553,1,C,26.0,NaN,NaN,5.0,211.5,1,1.126280,0.000000,0.307167,0,0,Jalen Duren,Cade Cunningham,Paul Reed,1,0,2,1,1,1,0,Paul Reed
27988,NaN,NaN,NaN,2025-26,1627747,Caris LeVert,Caris,1610612765,DET,Detroit Pistons,42500203,2026-05-09,DET @ CLE,L,17.200000,2,6,0.333,0,0,0.000,2,2,1.000,2,0,2,0,1,4,0,1,0,1,6,-10,19.4,0,0,16.0,1,17:12,1,85.0,88.9,88.9,118.6,110.5,110.5,-33.6,-21.6,-21.6,0.000,0.00,0.0,0.083,0.000,0.054,12.5,12.7,0.333,0.436,0.178,0.177,101.92,103.26,86.05,103.26,0.103,36,2.0,6.0,NaN,4.30,1.37,2.0,2.0,3.0,27.0,0.0,0.0,19.0,1.0,2.0,0.500,1.0,4.0,0.250,0.0,0.0,0.00,41,91,0.451,9,25,0.360,18,22,0.818,17,23,40,23,16.0,12,4,7,25,17,109,-7.0,109.3,114.7,120.4,119.6,-11.1,-4.9,0.561,1.44,16.5,0.404,0.722,0.534,0.168,0.500,0.541,98.0,96.0,80.00,95,0.447,1610612739,CLE,Cleveland Cavaliers,43,74,0.581,12,32,0.375,18,28,0.643,5,28,33,22,15.0,3,7,4,17,25,116,7.0,120.4,119.6,109.3,114.7,11.1,4.9,0.512,1.47,17.3,0.278,0.596,0.466,0.155,0.662,0.672,98.0,96.0,80.00,97,0.553,1,SG,31.0,NaN,NaN,5.0,211.5,1,0.348837,0.000000,0.116279,0,4,Jalen Duren,Cade Cunningham,Paul Reed,0,0,2,1,1,1,0,Caris LeVer

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260511_001318.json


,home_team,away_team,commence_time,bookmakers
0,Cleveland Cavaliers,Detroit Pistons,2026-05-12 00:00:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Los Angeles Lakers,Oklahoma City Thunder,2026-05-12 02:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,San Antonio Spurs,Minnesota Timberwolves,2026-05-13 02:00:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [6]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-11 00:13:32
US latest pull: 2026-05-11 00:13:19


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Cade Cunningham,Over,27.5,-137,2026-05-12,2026-05-11T07:12:38Z,2026-05-11 00:13:32
1,PrizePicks,player_points,Cade Cunningham,Under,27.5,-137,2026-05-12,2026-05-11T07:12:38Z,2026-05-11 00:13:32
2,PrizePicks,player_points,Donovan Mitchell,Over,27.5,-137,2026-05-12,2026-05-11T07:12:38Z,2026-05-11 00:13:32
3,PrizePicks,player_points,Donovan Mitchell,Under,27.5,-137,2026-05-12,2026-05-11T07:12:38Z,2026-05-11 00:13:32
4,PrizePicks,player_points,James Harden,Over,19.5,-137,2026-05-12,2026-05-11T07:12:38Z,2026-05-11 00:13:32


In [7]:
lines_us_pts.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,DraftKings,player_points,Cade Cunningham,Over,27.5,-105,2026-05-12,2026-05-11T07:13:11Z,2026-05-11 00:13:19
1,DraftKings,player_points,Cade Cunningham,Under,27.5,-122,2026-05-12,2026-05-11T07:13:11Z,2026-05-11 00:13:19
2,DraftKings,player_points,Donovan Mitchell,Over,27.5,-105,2026-05-12,2026-05-11T07:13:11Z,2026-05-11 00:13:19
3,DraftKings,player_points,Donovan Mitchell,Under,27.5,-121,2026-05-12,2026-05-11T07:13:11Z,2026-05-11 00:13:19
4,DraftKings,player_points,James Harden,Over,19.5,-107,2026-05-12,2026-05-11T07:13:11Z,2026-05-11 00:13:19


### Load my models

In [8]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-05-07.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-05-07.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-05-07.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-05-08.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [9]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
reb_preds.head(10)

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90
0,Jalen Duren,REB,26.17,30.24,36.81,0.1675,0.2939,0.4531,"[0.2134146341463415, 0.2841357537490134, 0.327...",4.38,8.89,16.68
1,Jarrett Allen,REB,24.41,30.67,34.65,0.1460,0.2696,0.4211,"[0.2457577530719719, 0.1192289858912366, 0.151...",3.56,8.27,14.59
2,Evan Mobley,REB,29.85,35.98,39.34,0.1206,0.2292,0.3557,"[0.2121212121212121, 0.2394014962593516, 0.182...",3.60,8.25,13.99
3,Ausar Thompson,REB,23.24,28.95,35.68,0.0979,0.1942,0.3106,"[0.274869109947644, 0.2895927601809955, 0.2147...",2.28,5.62,11.08
4,Tobias Harris,REB,29.28,37.16,39.90,0.0835,0.1731,0.3041,"[0.1675197766402978, 0.3361344537815126, 0.182...",2.44,6.43,12.13
5,James Harden,REB,34.14,40.05,42.13,0.0582,0.1395,0.2385,"[0.0602560883755962, 0.1433349259436216, 0.120...",1.99,5.59,10.05
6,Donovan Mitchell,REB,32.50,38.28,40.38,0.0490,0.1175,0.2384,"[0.0968783638320775, 0.1890869800108049, 0.152...",1.59,4.50,9.62
7,Max Strus,REB,19.54,25.19,30.17,0.0755,0.1704,0.2959,"[0.1256106071179344, 0.1129943502824858, 0.231...",1.47,4.29,8.93
8,Duncan Robinson,REB,27.11,35.34,37.86,0.0362,0.1107,0.2181,"[0.0, 0.1879208644359764, 0.0, 0.0344867226117...",0.98,3.91,8.25
9,Isaiah Stewart II,REB,5.59,12.21,21.14,0.0621,0.1612,0.3008,"[0.2116402116402116, 0.3211991434689507, 0.241...",0.35,1.97,6.36


In [10]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
pts_preds.head()

Cade Cunningham [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 38.6→41.5 (Δ+2.90)  RATE: 0.7087→0.6810 (Δ-0.0277)
Donovan Mitchell [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 38.3→37.6 (Δ-0.73)  RATE: 0.7019→0.5390 (Δ-0.1629)
James Harden [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 40.0→38.9 (Δ-1.16)  RATE: 0.5183→0.4393 (Δ-0.0790)
Tobias Harris [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 37.2→42.2 (Δ+5.00)  RATE: 0.4690→0.4294 (Δ-0.0396)
Evan Mobley [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 36.0→36.2 (Δ+0.24)  RATE: 0.4353→0.4403 (Δ+0.0050)
Jalen Duren [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 30.2→32.9 (Δ+2.62)  RATE: 0.4278→0.1115 (Δ-0.3163)
Jarrett Allen [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 30.7→30.9 (Δ+0.27)  RATE: 0.4241→0.3969 (Δ-0.0272)
Duncan Robinson [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 35.3→34.6 (Δ-0.71)  RATE: 0.3750→0.3382 (Δ-0.0368)
Ausar Thompson [PTS] [ix: away_underdog]  pace

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Cade Cunningham,PTS,37.95,41.51,44.38,0.4289,0.6810,0.9098,"[0.940042, 0.694386, 0.628091, 0.614423, 1.003...",16.28,28.27,40.38,True,None,3,0,underdog,3.5,212.5,2.9050,-0.0277,2.91,-0.0277,True,"{'stars': (-0.0647, 88), 'pace': (1.2007, 47),...","{'stars': (0.025, 88), 'pace': (-0.0309, 47), ..."
1,Donovan Mitchell,PTS,31.77,37.55,39.65,0.2352,0.5390,0.7987,"[0.870469, 0.647473, 0.293258, 0.378616, 0.441...",7.47,20.24,31.67,True,None,3,0,favorite,-3.5,212.5,-0.7300,-0.1629,-0.73,-0.1629,True,"{'stars': (-0.567, 91), 'pace': (-0.2651, 51),...","{'stars': (-0.0548, 91), 'pace': (-0.0989, 51)..."
2,James Harden,PTS,32.98,38.89,40.97,0.2611,0.4393,0.7058,"[0.583817, 0.723676, 0.46426, 0.432348, 0.4995...",8.61,17.08,28.92,True,None,3,0,favorite,-3.5,212.5,-1.1564,-0.0790,-1.16,-0.0790,True,"{'stars': (-0.4114, 54), 'pace': (0.1659, 54),...","{'stars': (-0.0251, 54), 'pace': (-0.0019, 54)..."
3,Tobias Harris,PTS,34.28,42.16,44.90,0.2100,0.4294,0.6935,"[0.435039, 0.449323, 0.558424, 0.534838, 0.690...",7.20,18.10,31.14,True,None,3,0,underdog,3.5,212.5,5.0000,-0.0396,5.00,-0.0396,True,"{'stars': (1.7811, 78), 'pace': (1.7721, 47), ...","{'stars': (-0.0189, 78), 'pace': (0.023, 47), ..."
4,Evan Mobley,PTS,30.09,36.22,39.58,0.2521,0.4403,0.7001,"[0.520152, 0.75313, 0.461158, 0.251343, 0.6943...",7.59,15.95,27.71,True,None,3,0,favorite,-3.5,212.5,0.2383,0.0050,0.24,0.0050,True,"{'stars': (-0.6202, 86), 'pace': (-0.0026, 51)...","{'stars': (-0.0065, 86), 'pace': (0.0151, 51),..."


### Get Line Probabilities

In [11]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
0,Cade Cunningham,AST,9.5,37.95,41.51,44.38,2.28,6.20,13.59,0.244,0.756
1,James Harden,AST,6.5,32.98,38.89,40.97,3.18,6.93,12.34,0.522,0.478
2,Donovan Mitchell,AST,4.0,31.77,37.55,39.65,0.00,1.99,6.49,0.185,0.815
3,LeBron James,AST,7.5,35.83,39.26,43.21,2.33,5.99,12.10,0.401,0.599
4,Austin Reaves,AST,6.5,35.33,40.66,42.29,1.54,5.13,9.80,0.442,0.558


In [13]:
import pandas as pd
from src.utils.generalized_best_bets_v2 import enrich_dfs_picks

dfs_df = pd.read_csv("data/raw/player_lines/NBA_DFS_20260510_213929.csv")
us_df  = pd.read_csv("data/raw/player_lines/NBA_US_20260510_213820.csv")

enriched_path, aligned_path, df = enrich_dfs_picks(
    dfs_df=dfs_df,
    us_df=us_df,
    base_df=base_df,                  # your existing game log df
    all_line_probs=all_line_probs,            # your existing all_line_probs df
    team_odds_source="data/props/circa+betonline_team_lines/circa+betonline_20260510_213347.json",
    dfs_platforms=None,  # or None for all four
    current_date="2026-05-10",
)


  DFS rows: 757 across ['Betr DFS', 'DraftKings Pick6', 'PrizePicks', 'Underdog']
  Sharp books available: ['Pinnacle', 'FanDuel', 'DraftKings', 'BetMGM', 'BetOnline.ag', 'Bovada']
  Fetching league team ratings (NBA API)...
  Enriched: 703 | sharp_verified: 103 | dfs_only: 0 | conflict: 101 | no_model: 499
  → data/props/enriched/dfs_enriched_20260510.json
  → data/props/enriched/dfs_sharp_aligned_20260510.json


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
7,Joel Embiid,AST,4.5,26.26,31.36,40.54,15.40,17.41,20.49,0.757,0.243,AST,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,4.9,6.0,2.85,0.4,1.5,-0.140,0.556,0.444,6.15,-13.31,0.8,0.6,0.53,0.40,34.24,4.76,0.34,0.04,4.00,4.0
28,Quentin Grimes,REB,2.5,15.05,21.94,27.71,7.52,10.15,10.91,0.732,0.268,REB,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,3.1,3.0,1.37,0.6,0.5,-0.438,0.669,0.331,27.72,-35.38,0.4,0.7,0.67,0.69,22.52,4.15,0.14,0.05,3.29,7.0
30,Rudy Gobert,REB,11.5,27.60,31.78,38.88,18.02,20.47,23.75,0.589,0.411,REB,Underdog,San Antonio Spurs,4.5,216.5,110.4,3.0,100.72,12.0,102.0,-115.0,0.495,0.535,11.2,11.0,3.01,-0.3,-0.5,0.100,0.460,0.540,-7.08,0.96,0.6,0.5,0.53,0.46,33.07,4.64,0.11,0.04,9.57,7.0
38,De'Aaron Fox,REB,3.5,27.44,32.93,37.80,12.94,14.55,13.92,0.569,0.431,REB,Underdog,Minnesota Timberwolves,-4.5,216.5,112.5,8.0,101.50,10.0,-115.0,100.0,0.535,0.500,3.5,3.5,2.07,0.0,0.0,0.000,0.500,0.500,-6.52,0.00,0.4,0.5,0.53,0.57,33.18,3.68,0.25,0.03,3.88,8.0
62,Joel Embiid,PTS,26.5,26.26,31.36,40.54,13.36,23.81,40.82,0.464,0.536,PTS,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-103.0,-110.0,0.507,0.524,26.9,27.5,7.46,0.4,1.0,-0.054,0.522,0.478,2.88,-8.75,0.4,0.5,0.53,0.55,34.24,4.76,0.34,0.04,22.75,4.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.head(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,Shai Gilgeous-Alexander,AST,4.5,31.48,37.38,40.08,19.72,23.02,23.74,0.951,0.049,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,100.0,108.0,0.500,0.481,7.4,7.5,2.07,2.9,3.0,-1.401,0.919,0.081,83.80,-83.15,1.0,0.9,0.87,0.80,33.04,5.57,0.32,0.04,7.71,7.0
1,Austin Reaves,AST,3.5,32.59,37.79,41.92,17.38,19.78,19.16,0.711,0.289,AST,PrizePicks,Oklahoma City Thunder,8.5,215.2,106.5,1.0,100.37,16.0,-115.0,110.0,0.535,0.476,5.1,5.0,2.42,1.6,1.5,-0.661,0.746,0.254,39.47,-46.66,0.6,0.8,0.87,0.71,34.86,5.12,0.26,0.04,4.00,7.0
2,Luguentz Dort,AST,1.5,21.03,27.97,33.23,10.66,13.52,14.66,0.687,0.313,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,-115.0,-115.0,0.535,0.535,1.5,1.5,1.35,0.0,0.0,0.000,0.500,0.500,-6.52,-6.52,0.6,0.5,0.40,0.38,23.03,2.75,0.12,0.03,1.86,7.0
3,Jalen Brunson,AST,6.5,30.83,34.60,40.53,17.65,19.39,21.15,0.629,0.371,AST,PrizePicks,Philadelphia 76ers,1.5,214.5,114.4,17.0,100.39,15.0,-137.0,-137.0,0.578,0.578,6.5,7.0,3.50,0.5,1.0,-0.143,0.557,0.443,-3.64,-23.36,0.4,0.6,0.67,0.56,34.74,3.70,0.32,0.05,5.25,8.0
4,Tyrese Maxey,AST,6.0,26.14,32.91,43.25,14.86,17.89,21.84,0.502,0.498,AST,PrizePicks,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-137.0,-137.0,0.578,0.578,5.4,5.5,2.46,-0.6,-0.5,0.244,0.404,0.596,-30.11,3.10,0.2,0.3,0.40,0.46,37.86,6.29,0.27,0.05,4.86,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
93,James Harden,PTS,18.5,31.47,37.88,42.27,9.94,20.40,36.28,0.537,0.463,PTS,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-122.0,-103.0,0.550,0.507,21.3,20.5,4.08,2.8,2.0,-0.686,0.754,0.246,37.20,-51.52,0.4,0.7,0.53,0.71,35.19,4.93,0.27,0.06,22.40,10.0
111,Ajay Mitchell,PTS,15.5,23.68,33.18,38.39,4.29,14.08,26.24,0.366,0.634,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-115.0,-106.0,0.535,0.515,11.5,9.5,4.74,-4.0,-6.0,0.844,0.199,0.801,-62.80,55.67,0.2,0.1,0.13,0.22,25.78,6.70,0.19,0.06,9.25,4.0
39,Evan Mobley,REB,8.5,25.64,33.17,39.57,3.52,7.64,13.62,0.503,0.497,REB,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-113.0,-103.0,0.531,0.507,9.1,7.5,4.23,0.6,-1.0,-0.142,0.556,0.444,4.80,-12.49,0.6,0.4,0.47,0.56,31.69,5.46,0.22,0.05,9.14,14.0
28,Julius Randle,REB,6.5,26.56,35.36,40.93,1.65,5.43,11.24,0.430,0.570,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-130.0,0.476,0.565,6.7,7.0,2.31,0.2,0.5,-0.087,0.535,0.465,12.35,-17.73,0.6,0.6,0.53,0.54,33.35,2.97,0.27,0.04,6.29,7.0
119,Cason Wallace,PTS,6.5,9.53,15.75,22.80,0.73,5.51,14.92,0.305,0.695,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-128.0,105.0,0.561,0.488,6.8,6.0,4.34,0.3,-0.5,-0.069,0.528,0.472,-5.95,-3.24,0.2,0.4,0.40,0.52,21.87,3.16,0.14,0.05,7.43,7.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,Quentin Grimes,REB,2.5,12.27,19.21,27.07,0.49,2.12,5.94,0.565,0.435,REB,DraftKings Pick6,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,130.0,-115.0,0.435,0.535,3.2,3.0,1.23,0.7,0.5,-0.569,0.715,0.285,64.45,-46.72,0.6,0.7,0.73,0.69,22.74,4.31,0.17,0.07,3.67,6.0
26,Rudy Gobert,REB,11.5,21.35,30.40,36.47,2.47,6.47,13.26,0.126,0.874,REB,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-116.0,0.476,0.537,10.9,11.0,3.28,-0.6,-0.5,0.183,0.427,0.573,-10.33,6.70,0.6,0.5,0.60,0.46,32.82,4.90,0.11,0.04,9.50,6.0
72,OG Anunoby,PTS,14.5,27.95,35.76,41.20,5.32,15.77,29.23,0.638,0.362,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,100.0,-130.0,0.500,0.565,19.7,20.0,8.90,5.2,5.5,-0.584,0.720,0.280,44.00,-50.46,0.8,0.7,0.67,0.64,33.30,7.34,0.18,0.05,18.29,7.0
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703,AST,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,129.0,-135.0,0.437,0.574,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-3.13,0.44,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
77,Anthony Edwards,PTS,20.5,16.76,23.86,32.55,2.64,9.12,23.30,0.103,0.897,PTS,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,115.0,-122.0,0.465,0.550,21.8,20.5,11.56,0.3,-1.0,-0.026,0.510,0.490,9.65,-10.84,0.6,0.5,0.53,0.72,30.48,7.93,0.31,0.04,28.00,7.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
51,Isaiah Hartenstein,REB,8.5,9.32,16.88,23.18,1.59,5.15,10.54,0.259,0.741,REB,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-137.0,-137.0,0.578,0.578,8.8,8.0,3.97,0.8,0.0,-0.202,0.580,0.420,0.34,-27.34,0.4,0.4,0.47,0.56,21.14,4.87,0.14,0.03,9.57,7.0
41,Ausar Thompson,REB,7.0,24.75,32.87,39.53,3.14,7.07,12.91,0.660,0.340,REB,Betr DFS,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,7.5,7.5,3.37,0.5,0.5,-0.148,0.559,0.441,-3.30,-23.71,0.8,0.5,0.40,0.23,30.03,5.87,0.14,0.04,7.92,13.0
44,Donovan Mitchell,REB,4.0,30.13,36.45,41.75,2.12,4.86,9.86,0.803,0.197,REB,PrizePicks,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,5.3,5.5,1.42,1.3,1.5,-0.915,0.820,0.180,41.85,-68.86,0.8,0.8,0.60,0.46,34.73,3.00,0.30,0.06,4.42,12.0
114,Marcus Smart,PTS,10.5,30.59,39.25,45.10,5.65,17.72,33.50,0.766,0.234,PTS,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,105.0,-115.0,0.488,0.535,11.4,10.0,7.28,0.9,-0.5,-0.124,0.549,0.451,12.54,-15.68,0.6,0.5,0.47,0.39,31.33,6.19,0.18,0.06,14.00,2.0
29,Naz Reid,REB,6.5,17.35,24.38,30.98,1.52,4.71,10.71,0.379,0.621,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,130.0,-140.0,0.435,0.583,6.3,7.0,2.36,-0.2,0.5,0.085,0.466,0.534,7.18,-8.46,0.8,0.6,0.53,0.39,24.88,4.94,0.21,0.03,5.57,7.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 87  |  Pairs: 885  |  Slate: 10  |  STRONG: 1  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 26  |  Pairs: 105  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 33  |  Pairs: 90  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 77  |  Pairs: 655  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 10  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 87  |  Triples: 18373  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 26  |  Triples: 623  |  Slate: 5  |  STRONG: 0  |  MARGINAL: 5  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 77  |  Triples: 12278  |  Slate: 9  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 33  |  Triples: 502  |  Slate: 4  |  STRONG: 0  |  MARGINAL: 3  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
